# Bike Sharing Demand Forecasting

*Joshua Ren 3041881898*

*May 07, 2026*

---


In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from data import load_raw_hourly, data_overview
from eda import basic_summary
from preprocessing import prepare_hourly_data, chronological_split, split_summary

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)

---

## 1. Data Loading and Preprocessing

This project uses the hourly file from the Bike Sharing Dataset on Kaggle. The data cover Capital Bikeshare rentals in Washington, DC from January 1, 2011 through December 31, 2012. Each row records the total number of rentals in one hour, together with calendar and weather information. The response variable is `cnt`, the aggregate hourly rental count.

**Variables in `hour.csv`**

| Group | Variable | Type | Description | Values / units |
|---|---|---|---|---|
| Time and calendar | `instant` | int | Record index in the original file | 1-17,379 |
| Time and calendar | `dteday` | date | Calendar date of the observation | YYYY-MM-DD |
| Time and calendar | `season` | int | Season indicator | 1: spring; 2: summer; 3: fall; 4: winter |
| Time and calendar | `yr` | int | Year indicator | 0: 2011; 1: 2012 |
| Time and calendar | `mnth` | int | Month of year | 1-12 |
| Time and calendar | `hr` | int | Hour of day | 0-23 |
| Time and calendar | `holiday` | int | Holiday indicator | 1: yes; 0: no |
| Time and calendar | `weekday` | int | Day of week | 0-6 |
| Time and calendar | `workingday` | int | Working-day indicator | 1: yes; 0: no |
| Weather | `weathersit` | int | Weather condition | 1: clear; 2: mist/cloudy; 3: light snow/rain; 4: heavy rain/snow |
| Weather | `temp` | float | Normalized temperature | scaled to [0, 1] |
| Weather | `atemp` | float | Normalized feeling temperature | scaled to [0, 1] |
| Weather | `hum` | float | Normalized humidity | scaled to [0, 1] |
| Weather | `windspeed` | float | Normalized wind speed | scaled to [0, 1] |
| Demand | `casual` | int | Count of non-registered users | hourly rental count |
| Demand | `registered` | int | Count of registered users | hourly rental count |
| Demand | `cnt` | int | Total bike rental demand | hourly rental count |

`cnt` is used as the response variable. `casual` and `registered` are not used as predictors because they add up to `cnt`.

### 1.1 Raw Data Overview

The first step loads `hour.csv`, creates an hourly timestamp, and checks the basic information of the raw data.

In [2]:
raw_df = load_raw_hourly()
data_overview(raw_df)

rows                                  17379
start_timestamp         2011-01-01 00:00:00
end_timestamp           2012-12-31 23:00:00
duplicate_timestamps                      0
expected_hours                        17544
observed_hours                        17379
missing_hours                           165
dtype: object

The file contains 17,379 hourly records. There are no duplicate timestamps, but the full hourly range should contain 17,544 hours, so 165 timestamps are missing and need to be handled before modeling.

### 1.2 Preprocess Hourly Data

The preprocessing step turns the raw file into a complete hourly series for modeling. It

- Reindex to the full hourly range so missing timestamps become rows.
- Extract calendar fields from the timestamp.
- Interpolate continuous variables.
- Keep only variables used later in the analysis.

In [3]:
model_df = prepare_hourly_data(raw_df)
model_df.head()

,cnt,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,hum,windspeed
timestamp,,,,,,,,,,,,
2011-01-01 00:00:00,16.0,1,0,1,0,0,6,0,1.0,0.24,0.81,0.0
2011-01-01 01:00:00,40.0,1,0,1,1,0,6,0,1.0,0.22,0.80,0.0
2011-01-01 02:00:00,32.0,1,0,1,2,0,6,0,1.0,0.22,0.80,0.0
2011-01-01 03:00:00,13.0,1,0,1,3,0,6,0,1.0,0.24,0.75,0.0
2011-01-01 04:00:00,1.0,1,0,1,4,0,6,0,1.0,0.24,0.75,0.0


## 2. Exploratory Data Analysis

Move the final exploratory plots and ACF/PACF checks here after they settle in `exploratory_pipeline.ipynb`.

In [4]:
workingday_summary, weather_corr = basic_summary(model_df)
workingday_summary, weather_corr

(              mean_cnt  median_cnt
 workingday                        
 0           180.469517       119.0
 1           191.313125       149.0,
 cnt          1.000000
 temp         0.409319
 windspeed    0.086004
 hum         -0.325274
 Name: cnt, dtype: float64)

## 3. Models

This section will compare the ARIMA benchmark, lagged regression, LSTM, RNN, and optional ARIMAX extension.

In [5]:
train_df, valid_df, test_df = chronological_split(model_df)
split_summary(train_df, valid_df, test_df)

,rows,start,end
train,10526,2011-01-01 00:00:00,2012-03-14 13:00:00
validation,3509,2012-03-14 14:00:00,2012-08-07 18:00:00
test,3509,2012-08-07 19:00:00,2012-12-31 23:00:00


## 4. Diagnostics and Interpretation

Add residual diagnostics, stationarity checks, the final comparison table, and the short interpretation that connects back to the project question.